# EXHEART 14 — Retrained BRFSS 2020 pipeline with corrected encoding

**Why.** The principal retrained-2020 results were computed with `LabelEncoder`, which (a) imposes an invalid ordinal scale on the nominal Race variable and (b) scrambles the GenHealth health scale into alphabetical order. The paper's own transport analysis shows encoding is consequential (0.122 sensitivity shift). This notebook re-encodes the 2020 pipeline correctly and regenerates every affected result so the manuscript can adopt them as primary:

- **One-hot Race** (nominal — no valid ordinal encoding), **semantic GenHealth** (Excellent→Poor), **ordered AgeCategory**, **documented Diabetic** and binary mappings, plus a published encoding/harmonisation table.
- Regenerates performance, calibration, meta-learner coefficients, Sex/Age/Race fairness, the **race inclusion ablation with one-hot as primary**, intersectional Sex×Age, and SHAP rankings.
- **Quantifies the change** versus the old LabelEncoder scheme, so the effect of the correction is explicit.

Outputs to `results/brfss2020_reencoded/`. Runtime is substantial (three full stack trainings on ~320k rows; budget 20–30 min).

### 1. Determinism

In [1]:
import os
os.environ['PYTHONHASHSEED']='42'; os.environ['TF_DETERMINISTIC_OPS']='1'; os.environ['TF_CUDNN_DETERMINISTIC']='1'
import random, numpy as np, tensorflow as tf
SEED=42; random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
try: tf.config.experimental.enable_op_determinism()
except Exception as e: print(e)

### 2. Mount, git, paths

In [2]:
from google.colab import drive
drive.mount('/content/drive')
import shutil, json, joblib, warnings
import pandas as pd, matplotlib.pyplot as plt
warnings.filterwarnings('ignore')
DRIVE_ROOT='/content/drive/MyDrive/EXHEART_Research'; REPO_DIR=os.path.join(DRIVE_ROOT,'exheart-research')
for fn in ['.git-credentials','.gitconfig']:
    s=os.path.join(DRIVE_ROOT,fn)
    if os.path.exists(s): shutil.copy(s, os.path.join('/root',fn)); print('restored',fn)
DATA_2020=os.path.join(REPO_DIR,'data/brfss2020/heart_2020_cleaned.csv')
RES=os.path.join(REPO_DIR,'results/brfss2020_reencoded'); os.makedirs(RES+'/figures',exist_ok=True); os.makedirs(RES+'/tables',exist_ok=True)
print('ready')

Mounted at /content/drive
ready


### 3. Libraries

In [3]:
!pip install -q scikit-learn xgboost lightgbm shap
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix, brier_score_loss
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from tensorflow import keras
from tensorflow.keras import layers
import shap

### 4. Encoding schemes and the harmonisation table

`encode_proper` gives every variable a defensible representation; `encode_naive` reproduces the original `LabelEncoder` scheme for comparison. The documented mappings are written to a table.

In [4]:
df=pd.read_csv(DATA_2020); TGT='HeartDisease'
y=(df[TGT]=='Yes').astype(int).values
YESNO={'No':0,'Yes':1}
AGE=['18-24','25-29','30-34','35-39','40-44','45-49','50-54','55-59','60-64','65-69','70-74','75-79','80 or older']
AGEMAP={a:i+1 for i,a in enumerate(AGE)}
GEN={'Excellent':1,'Very good':2,'Good':3,'Fair':4,'Poor':5}   # semantic health scale
DIAB={'No':0,'No, borderline diabetes':0,'Yes':1,'Yes (during pregnancy)':1}  # documented
BIN_COLS=['Smoking','AlcoholDrinking','Stroke','DiffWalking','PhysicalActivity','Asthma','KidneyDisease','SkinCancer']
NUM_COLS=['BMI','PhysicalHealth','MentalHealth','SleepTime']
RACE_COL='Race'

def encode_proper(d, include_race=True):
    cols={}
    for c in NUM_COLS: cols[c]=d[c].astype(float).values
    for c in BIN_COLS: cols[c]=d[c].map(YESNO).values
    cols['Sex']=d['Sex'].map({'Female':0,'Male':1}).values
    cols['AgeCategory']=d['AgeCategory'].map(AGEMAP).values
    cols['GenHealth']=d['GenHealth'].map(GEN).values
    cols['Diabetic']=d['Diabetic'].map(DIAB).values
    X=pd.DataFrame(cols)
    if include_race:
        oh=pd.get_dummies(d[RACE_COL], prefix='Race').astype(int)   # one-hot, nominal
        X=pd.concat([X.reset_index(drop=True), oh.reset_index(drop=True)], axis=1)
    return X

def encode_naive(d):
    X=pd.DataFrame({c:d[c].astype(float).values for c in NUM_COLS})
    for c in BIN_COLS+['Sex','AgeCategory','GenHealth','Diabetic',RACE_COL]:
        X[c]=LabelEncoder().fit_transform(d[c].astype(str))   # alphabetical, as in the original pipeline
    return X

# documented encoding / harmonisation table
tab=[]
for a,i in AGEMAP.items(): tab.append(('AgeCategory',a,i,'ordinal (increasing age)'))
for gname,gv in GEN.items(): tab.append(('GenHealth',gname,gv,'ordinal (health scale, Excellent->Poor)'))
for dname,dv in DIAB.items(): tab.append(('Diabetic',dname,dv,'documented binary'))
for c in BIN_COLS+['Sex']: tab.append((c,'No/Female -> 0, Yes/Male -> 1','', 'verified binary direction'))
tab.append(('Race', 'each category -> indicator column', '', 'one-hot (nominal, no valid ordinal)'))
enc_table=pd.DataFrame(tab, columns=['feature','category','code','encoding'])
enc_table.to_csv(RES+'/tables/encoding_harmonisation_table.csv', index=False)
print('race categories:', sorted(df[RACE_COL].unique()))
print('proper feature count:', encode_proper(df).shape[1], '| naive:', encode_naive(df).shape[1])

race categories: ['American Indian/Alaskan Native', 'Asian', 'Black', 'Hispanic', 'Other', 'White']
proper feature count: 22 | naive: 17


### 5. Leakage-safe stacked pipeline (XGB, LGBM, RF, MLP → LR meta → Platt)

In [5]:
def bases(posw):
    return {'xgb':XGBClassifier(n_estimators=300,max_depth=6,learning_rate=0.05,scale_pos_weight=posw,eval_metric='logloss',random_state=SEED,n_jobs=-1),
            'lgbm':LGBMClassifier(n_estimators=300,max_depth=6,learning_rate=0.05,class_weight='balanced',random_state=SEED,n_jobs=-1,verbose=-1),
            'rf':RandomForestClassifier(n_estimators=200,max_depth=10,class_weight='balanced',random_state=SEED,n_jobs=-1)}
def mkmlp(d):
    inp=keras.Input(shape=(d,)); x=layers.Dense(256,activation='relu')(inp); x=layers.BatchNormalization()(x); x=layers.Dropout(0.3)(x)
    x=layers.Dense(128,activation='relu')(x); x=layers.BatchNormalization()(x); x=layers.Dropout(0.3)(x)
    x=layers.Dense(64,activation='relu')(x); x=layers.Dropout(0.2)(x); o=layers.Dense(1,activation='sigmoid')(x)
    m=keras.Model(inp,o); m.compile(optimizer=keras.optimizers.Adam(1e-3),loss='binary_crossentropy',metrics=[keras.metrics.AUC(name='auc')]); return m

def train_stack(Xtr,ytr):
    Xtr=Xtr.values.astype(float); posw=(ytr==0).sum()/(ytr==1).sum()
    cw=compute_class_weight('balanced',classes=np.array([0,1]),y=ytr); kcw={0:cw[0],1:cw[1]}
    sc=StandardScaler().fit(Xtr)
    skf=StratifiedKFold(5,shuffle=True,random_state=SEED); oof=np.zeros((len(Xtr),4))
    for tri,vai in skf.split(Xtr,ytr):
        b=bases(posw)
        b['xgb'].fit(Xtr[tri],ytr[tri]); oof[vai,0]=b['xgb'].predict_proba(Xtr[vai])[:,1]
        b['lgbm'].fit(Xtr[tri],ytr[tri]); oof[vai,1]=b['lgbm'].predict_proba(Xtr[vai])[:,1]
        b['rf'].fit(Xtr[tri],ytr[tri]); oof[vai,2]=b['rf'].predict_proba(Xtr[vai])[:,1]
        s2=StandardScaler().fit(Xtr[tri]); tf.random.set_seed(SEED); mm=mkmlp(Xtr.shape[1])
        mm.fit(s2.transform(Xtr[tri]),ytr[tri],epochs=40,batch_size=1024,validation_split=0.1,class_weight=kcw,verbose=0,
               callbacks=[keras.callbacks.EarlyStopping(monitor='val_auc',patience=5,restore_best_weights=True,mode='max')])
        oof[vai,3]=mm.predict(s2.transform(Xtr[vai]),verbose=0).ravel()
    meta=LogisticRegression(max_iter=1000).fit(oof,ytr)
    bfull=bases(posw); [bfull[k].fit(Xtr,ytr) for k in bfull]
    tf.random.set_seed(SEED); mlp=mkmlp(Xtr.shape[1])
    mlp.fit(sc.transform(Xtr),ytr,epochs=40,batch_size=1024,validation_split=0.1,class_weight=kcw,verbose=0,
            callbacks=[keras.callbacks.EarlyStopping(monitor='val_auc',patience=5,restore_best_weights=True,mode='max')])
    return dict(bases=bfull,mlp=mlp,sc=sc,meta=meta)

def stack_raw(mdl,X):
    X=X.values.astype(float)
    return mdl['meta'].predict_proba(np.column_stack([mdl['bases']['xgb'].predict_proba(X)[:,1],mdl['bases']['lgbm'].predict_proba(X)[:,1],
        mdl['bases']['rf'].predict_proba(X)[:,1],mdl['mlp'].predict(mdl['sc'].transform(X),verbose=0).ravel()]))[:,1]

def fit_platt(mdl,Xcal,ycal): return LogisticRegression(max_iter=1000).fit(stack_raw(mdl,Xcal).reshape(-1,1),ycal)
def stack_cal(mdl,platt,X): return platt.predict_proba(stack_raw(mdl,X).reshape(-1,1))[:,1]

### 6. Metrics helpers

In [6]:
PT=0.12
def sens_spec(y,p,t=PT):
    yp=(p>=t).astype(int); tn,fp,fn,tp=confusion_matrix(y,yp,labels=[0,1]).ravel()
    return tp/(tp+fn), tn/(tn+fp)
def ece(y,p,bins=10):
    e=0
    for i in range(bins):
        lo,hi=i/bins,(i+1)/bins; m=(p>=lo)&(p<hi)
        if m.sum()>0: e+=abs(y[m].mean()-p[m].mean())*m.sum()/len(p)
    return e
def tpr_by(y,p,grp,t=PT):
    yp=(p>=t).astype(int); out={}
    for g in np.unique(grp):
        mm=(grp==g)&(y==1)
        out[g]=yp[mm].mean() if mm.sum()>0 else np.nan
    return out
def perf(y,p):
    se,sp=sens_spec(y,p)
    return dict(AUC=roc_auc_score(y,p),AUPRC=average_precision_score(y,p),sensitivity=se,specificity=sp,
                ECE=ece(y,p),Brier=brier_score_loss(y,p))

### 7. Train the corrected (proper-encoded) retrained-2020 model

In [7]:
Xp=encode_proper(df, include_race=True)
Xtr,Xte,ytr,yte,dtr,dte=train_test_split(Xp,y,df,test_size=0.2,random_state=SEED,stratify=y)
# carve a calibration holdout from train for Platt
Xtr2,Xcal,ytr2,ycal=train_test_split(Xtr,ytr,test_size=0.2,random_state=SEED,stratify=ytr)
mdl_p=train_stack(Xtr2,ytr2); platt_p=fit_platt(mdl_p,Xcal,ycal)
p_p=stack_cal(mdl_p,platt_p,Xte)
print('PROPER-encoded retrained 2020:'); print({k:round(v,4) for k,v in perf(yte,p_p).items()})
print('meta coefficients [xgb,lgbm,rf,mlp]:', np.round(mdl_p['meta'].coef_[0],3))

PROPER-encoded retrained 2020:
{'AUC': np.float64(0.8406), 'AUPRC': np.float64(0.3526), 'sensitivity': np.float64(0.604), 'specificity': np.float64(0.861), 'ECE': np.float64(0.0079), 'Brier': np.float64(0.0661)}
meta coefficients [xgb,lgbm,rf,mlp]: [0.028 1.275 0.887 3.322]


### 8. Fairness — Sex, Age, Race (one-hot model), intersectional

In [8]:
sex=dte['Sex'].values; race=dte['Race'].values; age=dte['AgeCategory'].values
st=tpr_by(yte,p_p,sex); sex_gap=abs(st['Male']-st['Female'])
rt=tpr_by(yte,p_p,race); race_gap=max(rt.values())-min(rt.values())
print(f'Sex TPR gap = {sex_gap:.3f}  (F={st["Female"]:.3f}, M={st["Male"]:.3f})')
print('Race TPR by group:', {k:round(v,3) for k,v in rt.items()}); print(f'Race TPR gap = {race_gap:.3f}')
# intersectional Sex x Age band (>=30 positives)
inter={}
for s in np.unique(sex):
    for a in np.unique(age):
        m=(sex==s)&(age==a); pos=(m)&(yte==1)
        if pos.sum()>=30: inter[(s,a)]=(p_p[pos]>=PT).mean()
if inter:
    vals=list(inter.values()); print(f'Sex x Age max TPR gap = {max(vals)-min(vals):.3f} (over {len(inter)} cells >=30 pos)')
json.dump({'sex_gap':float(sex_gap),'race_gap':float(race_gap),'race_tpr':{k:float(v) for k,v in rt.items()},
           'sex_tpr':{k:float(v) for k,v in st.items()}}, open(RES+'/tables/fairness_proper.json','w'), indent=2)

Sex TPR gap = 0.167  (F=0.506, M=0.674)
Race TPR by group: {'American Indian/Alaskan Native': np.float64(0.678), 'Asian': np.float64(0.492), 'Black': np.float64(0.539), 'Hispanic': np.float64(0.461), 'Other': np.float64(0.543), 'White': np.float64(0.62)}
Race TPR gap = 0.218
Sex x Age max TPR gap = 0.866 (over 19 cells >=30 pos)


### 9. Race inclusion ablation — one-hot as primary (with vs without race), bootstrap CI

In [9]:
# retrain WITHOUT race features (race still used only to group for the gap)
Xnr=encode_proper(df, include_race=False)
Xtr_nr=Xnr.loc[Xtr.index]; Xtr2_nr=Xnr.loc[Xtr2.index]; Xcal_nr=Xnr.loc[Xcal.index]; Xte_nr=Xnr.loc[Xte.index]
mdl_nr=train_stack(Xtr2_nr,ytr2); platt_nr=fit_platt(mdl_nr,Xcal_nr,ycal)
p_nr=stack_cal(mdl_nr,platt_nr,Xte_nr)
def race_gap_of(p):
    t=tpr_by(yte,p,race); return max(t.values())-min(t.values())
g_with=race_gap_of(p_p); g_without=race_gap_of(p_nr); diff=g_with-g_without
# bootstrap the difference
rng=np.random.default_rng(SEED); diffs=[]
for _ in range(1000):
    idx=rng.integers(0,len(yte),len(yte))
    yb,rb=yte[idx],race[idx]
    def gap(p):
        t={};
        for g in np.unique(rb):
            mm=(rb==g)&(yb==1)
            t[g]=(p[idx][mm]>=PT).mean() if mm.sum()>0 else np.nan
        v=[x for x in t.values() if x==x]; return (max(v)-min(v)) if v else np.nan
    d=gap(p_p)-gap(p_nr)
    if d==d: diffs.append(d)
lo,hi=np.percentile(diffs,[2.5,97.5])
print(f'ONE-HOT race ablation (PRIMARY):')
print(f'  race TPR gap WITH race feature   = {g_with:.3f}')
print(f'  race TPR gap WITHOUT race feature= {g_without:.3f}')
print(f'  difference = {diff:+.3f}  95% CI [{lo:+.3f}, {hi:+.3f}]  ({"excludes" if lo>0 or hi<0 else "straddles"} zero)')
json.dump({'gap_with':float(g_with),'gap_without':float(g_without),'difference':float(diff),'ci':[float(lo),float(hi)]},
          open(RES+'/tables/race_ablation_onehot.json','w'), indent=2)

ONE-HOT race ablation (PRIMARY):
  race TPR gap WITH race feature   = 0.218
  race TPR gap WITHOUT race feature= 0.141
  difference = +0.076  95% CI [-0.015, +0.132]  (straddles zero)


### 10. SHAP rankings on the corrected model

In [10]:
expl=shap.TreeExplainer(mdl_p['bases']['xgb'])
Xs=Xte.sample(min(2000,len(Xte)),random_state=SEED)
sv=expl.shap_values(Xs.values.astype(float))
if isinstance(sv,list): sv=sv[1]
imp=pd.Series(np.abs(sv).mean(0), index=Xp.columns).sort_values(ascending=False)
print('Top-10 features (mean |SHAP|, proper encoding):'); print(imp.head(10).round(4))
imp.to_csv(RES+'/tables/shap_proper.csv')

Top-10 features (mean |SHAP|, proper encoding):
AgeCategory       0.9898
GenHealth         0.5136
Sex               0.3185
Smoking           0.1825
Diabetic          0.1501
Stroke            0.1212
BMI               0.1103
DiffWalking       0.0977
PhysicalHealth    0.0825
Asthma            0.0644
dtype: float32


### 11. Comparison: corrected vs original LabelEncoder encoding

In [ ]:
Xn=encode_naive(df)
Xtr_n=Xn.loc[Xtr.index]; Xtr2_n=Xn.loc[Xtr2.index]; Xcal_n=Xn.loc[Xcal.index]; Xte_n=Xn.loc[Xte.index]
mdl_n=train_stack(Xtr2_n,ytr2); platt_n=fit_platt(mdl_n,Xcal_n,ycal)
p_n=stack_cal(mdl_n,platt_n,Xte_n)
rows=[]
for name,p in [('proper (one-hot race, semantic ordinals)',p_p),('naive (LabelEncoder)',p_n)]:
    m=perf(yte,p); st_=tpr_by(yte,p,sex); rt_=tpr_by(yte,p,race)
    rows.append(dict(encoding=name,**{k:round(v,4) for k,v in m.items()},
                     sex_gap=round(abs(st_['Male']-st_['Female']),3),
                     race_gap=round(max(rt_.values())-min(rt_.values()),3)))
comp=pd.DataFrame(rows); comp.to_csv(RES+'/tables/encoding_comparison.csv',index=False)
print(comp.to_string(index=False))
print('\nInterpretation: the delta between rows is the effect of correcting the encoding on the retrained-2020 results.')

### 12. Figure + commit

In [ ]:
fig,ax=plt.subplots(1,2,figsize=(12,4.5))
imp.head(8)[::-1].plot.barh(ax=ax[0],color='steelblue'); ax[0].set_title('SHAP top-8 (corrected encoding)'); ax[0].set_xlabel('mean |SHAP|')
grp=comp.set_index('encoding')[['AUC','sensitivity','sex_gap','race_gap']]
grp.T.plot.bar(ax=ax[1]); ax[1].set_title('Corrected vs naive encoding'); ax[1].tick_params(axis='x',rotation=0); ax[1].legend(fontsize=7)
plt.tight_layout(); plt.savefig(RES+'/figures/reencoded_summary.png',dpi=150,bbox_inches='tight'); plt.show()

In [ ]:
%cd {REPO_DIR}
!git add results/brfss2020_reencoded notebooks/EXHEART_14_reencoded_2020.ipynb 2>/dev/null
!git commit -m "Retrain BRFSS 2020 with corrected encoding (one-hot race primary, semantic ordinals) and quantify vs LabelEncoder"
!git push origin main